# Agentic RAG: Router-Retriever System with PDF and Web Search Tools

A multi-agent Retrieval-Augmented Generation (RAG) system built with **CrewAI**. A **Router Agent** classifies each user question, and a **Retriever Agent** answers it from the right source: a PDF knowledge base (NIST AI RMF 1.0), a live web search (Tavily), or the LLM's general knowledge. Every run is logged so the reasoning can be traced.

### Why CrewAI

- **Role-based design fits the problem.** The assignment is defined in terms of roles (router, retriever). CrewAI's `Agent(role, goal, backstory)` maps directly onto that, so each agent's responsibility is explicit and readable.
- **Built-in task hand-off.** `Task(context=[...])` passes one agent's output to the next without custom plumbing, which keeps the input-output flow clear.
- **RAG tools included.** `crewai_tools.PDFSearchTool` handles chunking, embedding, and vector search in one line.
- **Interoperability.** LangChain tools such as `TavilySearchResults` can be used through a small `BaseTool` wrapper.
- **Traceability.** `verbose=True`, `task_callback`, and `output_log_file` provide reasoning logs with little extra code.

**Alternatives considered:** LangGraph offers finer control over branching (an explicit graph edge per route) but takes more code to set up. Plain LangChain lacks first-class multi-agent roles. For a two-agent sequential workflow, CrewAI gives the clearest design with the least code.

---

## 1. Setup

Load the API keys (`OPENAI_API_KEY`, `TAVILY_API_KEY`) from the `.env` file and confirm that the NIST PDF is in the project folder. The check prints only whether each key was found, never the key itself.

In [1]:
# Load API keys from the .env file into this notebook's environment
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env and makes the keys available to the libraries

# Check that each key was found, without printing the secret itself
for key_name in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    status = "found" if os.getenv(key_name) else "MISSING"
    print(f"{key_name}: {status}")

# Check that the PDF is where the notebook expects it
print("PDF found:", os.path.exists("nist_ai_rmf.pdf"))

OPENAI_API_KEY: found
TAVILY_API_KEY: found
PDF found: True


## 2. Tools

The Retriever Agent can use two tools.

**PDF search tool.** `PDFSearchTool` splits the NIST AI RMF PDF into chunks, turns each chunk into an embedding (a numeric representation of its meaning), and stores them in a local vector database. A query returns the passages closest in meaning, with page numbers. The test below queries the PDF directly, before any agents are involved.

In [2]:
# PDF search tool: turns the NIST PDF into a searchable "index of meanings"
from crewai_tools import PDFSearchTool

pdf_tool = PDFSearchTool(pdf="nist_ai_rmf.pdf")

# Quick test: ask the PDF a question directly, with no agents involved yet
result = pdf_tool.run(query="What are the four core functions of the AI RMF?")
print(result[:1500])  # show only the first 1,500 characters

Relevant Content:




Page 24:

NIST AI 100-1

AI RMF 1.0

4.

Effectiveness of the AI RMF

Evaluations of AI RMF effectiveness – including ways to measure bottom-line improve-

ments in the trustworthiness of AI systems – will be part of future NIST activities, in

conjunction with the AI community.

Organizations and other users of the Framework are encouraged to periodically evaluate

whether the AI RMF has improved their ability to manage AI risks, including but not lim-

ited to their policies, processes, practices, implementation plans, indicators, measurements,

and expected outcomes. NIST intends to work collaboratively with others to develop met-

rics, methodologies, and goals for evaluating the AI RMF’s effectiveness, and to broadly

share results and supporting information. Framework users are expected to benefit from:

• enhanced processes for governing, mapping, measuring, and managing AI risk, and

clearly documenting outcomes;

• improved awareness of the relationships 

**Web search tool.** Tavily is a LangChain tool, not a CrewAI tool. The small `WebSearchTool` class wraps it so CrewAI agents can call it, and formats each of the top 3 results with its source URL so answers can cite where information came from.

In [3]:
# Web search tool: Tavily, wrapped so CrewAI agents can use it
from crewai.tools import BaseTool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(max_results=3)  # the LangChain Tavily tool

class WebSearchTool(BaseTool):
    name: str = "Web Search"
    description: str = (
        "Searches the internet for current, up-to-date information. "
        "Input should be a search query string."
    )

    def _run(self, query: str) -> str:
        results = tavily.invoke({"query": query})
        if isinstance(results, str):  # Tavily returns plain text if something went wrong
            return results
        # Format each result with its URL so the agent can cite sources
        return "\n\n".join(f"Source: {r['url']}\n{r['content']}" for r in results)

web_tool = WebSearchTool()

# Quick test
print(web_tool.run(query="latest news on NIST AI Risk Management Framework"))

/var/folders/7l/jsbxbcjn2sj9km24h5b1d3v00000gn/T/ipykernel_49562/1206003329.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
/var/folders/7l/jsbxbcjn2sj9km24h5b1d3v00000gn/T/ipykernel_49562/1206003329.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(max_results=3)  # the LangChain Tavily tool


Source: https://www.nist.gov/itl/ai-risk-management-framework
On March 30, 2023, NIST launched the Trustworthy and Responsible AI Resource Center, which will facilitate implementation of, and international alignment with, the AI RMF. Examples of how other organizations are building on and using the AI RMF can be found via the AIRC’s Use Case page.

On July 26, 2024, NIST releasedNIST-AI-_600-1, Artificial Intelligence Risk Management Framework: Generative Artificial Intelligence Profile_. The profile can help organizations identify unique risks posed by generative AI and proposes actions for generative AI risk management that best aligns with their goals and priorities. [...] On April 7, 2026, NIST released a concept note for an AI RMF Profile on Trustworthy AI in Critical Infrastructure. The profile will guide critical infrastructure operators towards specific risk management practices to consider when engaging AI-enabled capabilities.

To view public comments received on the previous

## 3. Agents

Both agents share one LLM: `gpt-4o-mini` at `temperature=0`, so routing decisions and answers are consistent.

- **Router Agent ("Query Router")** works like a library's front desk. It has no tools and never answers the question. It chooses one source and explains why:
  - **PDF:** content of NIST AI RMF 1.0 (AI risk concepts, trustworthy-AI characteristics, Govern/Map/Measure/Manage).
  - **WEB:** current or time-sensitive questions, or topics the PDF does not cover.
  - **LLM:** general knowledge that needs no source.
- **Retriever Agent ("Information Retriever")** follows the router's decision, uses the matching tool (or none for LLM), and writes a short answer that ends with a `Sources:` line.

Both agents use `allow_delegation=False`, so control flows one way only: Router to Retriever.

In [4]:
from crewai import Agent, LLM

# The shared "brain" both agents think with.
# temperature=0 makes answers consistent and less creative,
# which is what we want for routing decisions and factual answers.
llm = LLM(model="gpt-4o-mini", temperature=0)

# ---------- Router Agent: the receptionist ----------
router_agent = Agent(
    role="Query Router",
    goal=(
        "Read the user's question and decide the single best source to answer it: "
        "PDF, WEB, or LLM."
    ),
    backstory=(
        "You are the front desk of a research library. You never answer questions yourself; "
        "you decide where the answer lives.\n"
        "- PDF: the library holds one reference document, the NIST AI Risk Management "
        "Framework (AI RMF 1.0, NIST AI 100-1, published January 2023). Choose PDF for questions "
        "about its content: AI risk concepts, trustworthy AI characteristics, the Govern/Map/"
        "Measure/Manage functions, categories, and how organizations should use the framework.\n"
        "- WEB: choose WEB for anything current or time-sensitive (news, recent updates, events "
        "after January 2023, 'latest', 'this year') or for topics the document does not cover.\n"
        "- LLM: choose LLM only for general knowledge or simple questions that need no source, "
        "such as definitions of common terms or basic explanations."
    ),
    tools=[],              # the router only decides, so it gets no tools
    llm=llm,
    allow_delegation=False,
    verbose=True,          # print its reasoning (part of our trace)
)

# ---------- Retriever Agent: the librarian ----------
retriever_agent = Agent(
    role="Information Retriever",
    goal=(
        "Use the source chosen by the Query Router to retrieve information and write an "
        "accurate, concise answer that cites its sources."
    ),
    backstory=(
        "You are a meticulous research librarian. You follow the router's decision exactly: "
        "for PDF you use only the PDF search tool, for WEB you use only the Web Search tool, "
        "and for LLM you answer from general knowledge without tools. You never invent facts. "
        "You cite page numbers for PDF answers and URLs for web answers. If the retrieved "
        "material does not contain the answer, you say so."
    ),
    tools=[pdf_tool, web_tool],  # it can use both tools, but only uses the one it's told to
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

print("Agents created:", router_agent.role, "|", retriever_agent.role)

Agents created: Query Router | Information Retriever


## 4. Tasks and Crew

Each agent gets one task:

1. **`route_task`** (Router Agent) returns two lines: `ROUTE: <PDF|WEB|LLM>` and `REASON: <one sentence>`.
2. **`retrieve_task`** (Retriever Agent) receives the router's output through `context=[route_task]`. This is the explicit hand-off between the agents. It then retrieves from the chosen source and writes the answer.

The `Crew` runs the two tasks in order (`Process.sequential`). The `{question}` placeholder in each task description is filled in when the crew starts.

In [5]:
from crewai import Task, Crew, Process

# ---------- Task 1: Routing (done by the Router Agent) ----------
route_task = Task(
    description=(
        "Analyze this user question and decide which source should answer it.\n\n"
        "Question: {question}\n\n"
        "Choose exactly one: PDF, WEB, or LLM."
    ),
    expected_output=(
        "Exactly two lines in this format:\n"
        "ROUTE: <PDF or WEB or LLM>\n"
        "REASON: <one sentence explaining the choice>"
    ),
    agent=router_agent,
)

# ---------- Task 2: Retrieval + Answer (done by the Retriever Agent) ----------
retrieve_task = Task(
    description=(
        "Answer the user's question using the route chosen by the Query Router "
        "(provided to you as context).\n\n"
        "Question: {question}\n\n"
        "Rules:\n"
        "- ROUTE: PDF -> search the NIST AI RMF PDF with the PDF search tool.\n"
        "- ROUTE: WEB -> search the internet with the Web Search tool.\n"
        "- ROUTE: LLM -> answer from general knowledge without using any tool.\n"
        "Base your answer only on what you retrieved. Do not invent facts."
    ),
    expected_output=(
        "A clear, concise answer (1-3 short paragraphs), followed by a 'Sources:' line "
        "listing PDF page numbers, web URLs, or 'General knowledge (no retrieval)'."
    ),
    agent=retriever_agent,
    context=[route_task],  # <-- the hand-off: Task 1's output is given to Task 2
)

# ---------- The Crew: runs the tasks in order ----------
crew = Crew(
    agents=[router_agent, retriever_agent],
    tasks=[route_task, retrieve_task],
    process=Process.sequential,  # Task 1 first, then Task 2
    verbose=True,
)

print("Crew ready with", len(crew.tasks), "tasks")

Crew ready with 2 tasks


### Quick test

Jupyter already runs its own event loop, so calling `crew.kickoff()` here raises a `RuntimeError`. The notebook uses `await crew.kickoff_async(...)` instead, which starts the crew and waits for it to finish.

In [6]:
# Jupyter already has an event loop running, so we use the async version of kickoff
# and "await" it, meaning: start the crew and wait here until it finishes.
result = await crew.kickoff_async(
    inputs={"question": "What are the characteristics of trustworthy AI according to the NIST AI RMF?"}
)

print("\n" + "=" * 60)
print("ROUTER DECISION:\n", result.tasks_output[0].raw)
print("\nFINAL ANSWER:\n", result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dcde7a47-5f3a-4ca0-8dac-d5bb594315f4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What are the characteristics of trustworthy AI according to the NIST AI RMF?                         │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  ID: 787633f9-f33d-42da-a55f-9e3a7235a3b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What are the characteristics of trustworthy AI according to the NIST AI RMF?                         │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ROUTE: PDF                                                                                                     │
│  REASON: The question specifically asks about the characteristics of trustworthy AI as outlined in the NIST AI  │
│  Risk Management Framework, which is covered in the PDF document.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What are the characteristics of trustworthy AI according to the NIST AI RMF?                         │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (6 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the user's question using the route chosen by the Query Router (provided to you as context).      │
│                                                                                                                 │
│  Question: What are the characteristics of trustworthy AI according to the NIST AI RMF?                         │
│                                                                                                                 │
│  Rules:                                                                                                         │
│  - ROUTE: PDF -> search the NIST AI RMF PDF with the PDF search tool.                                           │
│  - ROUTE: WEB -> search the internet with the Web Search tool.                                                  │
│  - ROUTE: LLM -> answer from general knowledge without using any tool.                                          │
│  Base your answer only on what you retrieved. Do not invent facts.                                              │
│  Agent: Information Retriever                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


ROUTER DECISION:
 ROUTE: PDF  
REASON: The question specifically asks about the characteristics of trustworthy AI as outlined in the NIST AI Risk Management Framework, which is covered in the PDF document.

FINAL ANSWER:
 According to the NIST AI Risk Management Framework (AI RMF), the characteristics of trustworthy AI systems include the following: they must be valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed. These characteristics are essential for reducing negative AI risks and ensuring that AI systems are responsive to the needs of various stakeholders. It is important to balance these characteristics based on the specific context of use, as neglecting any of them can increase the likelihood of negative consequences.

The framework emphasizes that trustworthiness is a social concept that varies across a spectrum and is influenced by the interplay of these characteristics

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: dcde7a47-5f3a-4ca0-8dac-d5bb594315f4                                                                       │
│  Final Output: According to the NIST AI Risk Management Framework (AI RMF), the characteristics of trustworthy  │
│  AI systems include the following: they must be valid and reliable, safe, secure and resilient, accountable     │
│  and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed. These    │
│  characteristics are essential for reducing negative AI risks and ensuring that AI systems are responsive to    │
│  the needs of various stakeholders. It is important to balance these characteristics based on the specific      │
│  context of use, as neglecting any of them can increase the likelihood of negative consequences.                │
│                                                                                                                 │
│  The framework emphasizes that trustworthiness is a social concept that varies across a spectrum and is         │
│  influenced by the interplay of these characteristics. For instance, trade-offs may arise between               │
│  interpretability and privacy, or between predictive accuracy and interpretability. Therefore, a comprehensive  │
│  approach to managing AI risks involves considering these trade-offs and making informed decisions based on     │
│  the context and the values at play.                                                                            │
│                                                                                                                 │
│  Sources: NIST AI RMF, pages 17-18.                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 5. Logging

Every run is recorded so the agents' reasoning can be traced:

- **`trace_log.txt`**: a clean log written by our own logger. A `task_callback` adds each agent's output as soon as its task finishes, and `ask()` adds the question, the parsed route and reason, the final answer, and the elapsed time.
- **`crew_verbose_log.txt`**: CrewAI's full verbose trace, including tool calls (via `output_log_file`).
- **`trace_records`**: an in-memory list of each run, used for the summary table and trace view in section 7.

This cell also defines `ask(question)`, the single entry point used in the demo.

In [7]:
import logging
import time
from datetime import datetime
from IPython.display import Markdown, display

# ---------- 1. Set up our own logger that writes to trace_log.txt ----------
logger = logging.getLogger("agentic_rag")
logger.setLevel(logging.INFO)
logger.handlers.clear()                      # avoid duplicate lines if this cell is re-run
file_handler = logging.FileHandler("trace_log.txt", mode="a")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(message)s"))
logger.addHandler(file_handler)
logger.propagate = False                     # keep our log separate from CrewAI's own logging

trace_records = []   # in-notebook copy of every run, used for the summary table later


# ---------- 2. A callback: CrewAI calls this automatically after each task finishes ----------
def log_task_output(task_output):
    logger.info(f"AGENT FINISHED: {task_output.agent}")
    logger.info(f"AGENT OUTPUT: {task_output.raw[:800]}")   # first 800 characters


# ---------- 3. ask(): one question in, a routed and grounded answer out ----------
async def ask(question):
    logger.info("=" * 70)
    logger.info(f"QUESTION: {question}")
    start = time.time()

    crew = Crew(
        agents=[router_agent, retriever_agent],
        tasks=[route_task, retrieve_task],
        process=Process.sequential,
        verbose=True,
        task_callback=log_task_output,              # our flight recorder
        output_log_file="crew_verbose_log.txt",     # CrewAI's full detailed log
    )
    result = await crew.kickoff_async(inputs={"question": question})

    # Pull the route out of "ROUTE: PDF\nREASON: ..."
    router_text = result.tasks_output[0].raw
    route = router_text.split("ROUTE:")[1].split()[0] if "ROUTE:" in router_text else "UNKNOWN"
    reason = router_text.split("REASON:")[1].strip() if "REASON:" in router_text else ""
    elapsed = round(time.time() - start, 1)

    logger.info(f"ROUTE: {route} | REASON: {reason}")
    logger.info(f"FINAL ANSWER: {result.raw[:800]}")
    logger.info(f"ELAPSED: {elapsed}s")

    trace_records.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "question": question,
        "route": route,
        "reason": reason,
        "seconds": elapsed,
    })

    # Show a clean summary under all the verbose output
    display(Markdown(
        f"### Question\n{question}\n\n"
        f"**Route:** `{route}` — {reason}\n\n"
        f"### Answer\n{result.raw}\n\n*Completed in {elapsed}s*"
    ))
    return result

print("Logger and ask() ready. Traces will be written to trace_log.txt")

Logger and ask() ready. Traces will be written to trace_log.txt


## 6. Demo

Three questions, one for each route. The comment above each one gives the expected route, so the router's choice can be checked against it.

In [8]:
# Demo 1: should route to PDF (static, domain-specific content)
await ask("What does the GOVERN function cover in the NIST AI RMF?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 52dbab27-49aa-4105-bc2b-7406dcaf15a5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What does the GOVERN function cover in the NIST AI RMF?                                              │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  ID: 787633f9-f33d-42da-a55f-9e3a7235a3b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What does the GOVERN function cover in the NIST AI RMF?                                              │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ROUTE: PDF                                                                                                     │
│  REASON: The question specifically asks about the content of the NIST AI Risk Management Framework, which is    │
│  covered in the reference document.                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What does the GOVERN function cover in the NIST AI RMF?                                              │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (9 outputs left out)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 52dbab27-49aa-4105-bc2b-7406dcaf15a5                                                                       │
│  Final Output: The GOVERN function in the NIST AI Risk Management Framework (AI RMF) is designed to be a        │
│  cross-cutting function that informs and integrates throughout the other three functions: MAP, MEASURE, and     │
│  MANAGE. It encompasses the overarching governance processes that organizations should implement to manage AI   │
│  risks effectively. This function applies to all stages of an organization's AI risk management processes and   │
│  procedures, ensuring that governance is infused into every aspect of AI system management.                     │
│                                                                                                                 │
│  The AI RMF Core outlines that the GOVERN function includes specific categories and subcategories that guide    │
│  organizations in establishing policies, practices, and procedures for AI risk management. This function        │
│  emphasizes the importance of continuous and timely risk management throughout the AI system lifecycle,         │
│  reflecting diverse perspectives and fostering collaboration among various stakeholders involved in AI          │
│  development and deployment (NIST AI RMF, pp. 20-22).                                                           │
│                                                                                                                 │
│  Sources: NIST AI RMF, pp. 20-22. Available at: https://doi.org/10.6028/NIST.AI.100-1                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Question
What does the GOVERN function cover in the NIST AI RMF?

**Route:** `PDF` — The question specifically asks about the content of the NIST AI Risk Management Framework, which is covered in the reference document.

### Answer
The GOVERN function in the NIST AI Risk Management Framework (AI RMF) is designed to be a cross-cutting function that informs and integrates throughout the other three functions: MAP, MEASURE, and MANAGE. It encompasses the overarching governance processes that organizations should implement to manage AI risks effectively. This function applies to all stages of an organization's AI risk management processes and procedures, ensuring that governance is infused into every aspect of AI system management.

The AI RMF Core outlines that the GOVERN function includes specific categories and subcategories that guide organizations in establishing policies, practices, and procedures for AI risk management. This function emphasizes the importance of continuous and timely risk management throughout the AI system lifecycle, reflecting diverse perspectives and fostering collaboration among various stakeholders involved in AI development and deployment (NIST AI RMF, pp. 20-22).

Sources: NIST AI RMF, pp. 20-22. Available at: https://doi.org/10.6028/NIST.AI.100-1

*Completed in 10.3s*

CrewOutput(raw="The GOVERN function in the NIST AI Risk Management Framework (AI RMF) is designed to be a cross-cutting function that informs and integrates throughout the other three functions: MAP, MEASURE, and MANAGE. It encompasses the overarching governance processes that organizations should implement to manage AI risks effectively. This function applies to all stages of an organization's AI risk management processes and procedures, ensuring that governance is infused into every aspect of AI system management.\n\nThe AI RMF Core outlines that the GOVERN function includes specific categories and subcategories that guide organizations in establishing policies, practices, and procedures for AI risk management. This function emphasizes the importance of continuous and timely risk management throughout the AI system lifecycle, reflecting diverse perspectives and fostering collaboration among various stakeholders involved in AI development and deployment (NIST AI RMF, pp. 20-22).\n\nSo

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
# Demo 2: should route to WEB (current information the 2023 PDF can't have)
await ask("What updates or new profiles has NIST released for the AI RMF in 2025 or 2026?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fbc6b101-ec3a-4626-8079-9524b8ef7277                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What updates or new profiles has NIST released for the AI RMF in 2025 or 2026?                       │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  ID: 787633f9-f33d-42da-a55f-9e3a7235a3b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What updates or new profiles has NIST released for the AI RMF in 2025 or 2026?                       │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ROUTE: WEB                                                                                                     │
│  REASON: The question pertains to updates or new profiles released by NIST in 2025 or 2026, which are current   │
│  and time-sensitive topics not covered in the PDF.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: What updates or new profiles has NIST released for the AI RMF in 2025 or 2026?                       │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (7 outputs left out)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: fbc6b101-ec3a-4626-8079-9524b8ef7277                                                                       │
│  Final Output: In 2025 and 2026, NIST has introduced significant updates to the AI Risk Management Framework    │
│  (AI RMF), focusing on tailored profiles and deeper integration with existing cybersecurity and privacy         │
│  frameworks. Notably, the introduction of AI RMF Profiles aims to customize the framework for specific          │
│  sectors, including a concept note for a "Trustworthy AI Profile for Critical Infrastructure." This reflects    │
│  an evolving understanding of AI systems as critical components that require robust governance and risk         │
│  management.                                                                                                    │
│                                                                                                                 │
│  Additionally, NIST has aligned the AI RMF with the NIST Cybersecurity Framework (CSF 2.0) and the NIST         │
│  Privacy Framework, facilitating a more cohesive approach to enterprise risk management. The updates also       │
│  include operational resources such as playbooks and implementation guides, which provide organizations with    │
│  practical tools for measuring and managing AI risks effectively. A preliminary draft of the Cyber AI Profile   │
│  (NIST IR 8596) was released in December 2025, focusing on securing AI systems and addressing AI-enabled cyber  │
│  threats.                                                                                                       │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  -                                                                                                              │
│  https://www.ispartnersllc.com/blog/nist-ai-rmf-2025-2026-updates-what-you-need-to-know-about-the-latest-frame  │
│  work-changes                                                                                                   │
│  - https://www.ampcuscyber.com/blogs/nist-csf-20-ai-rmf-us-implementation-guide                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Question
What updates or new profiles has NIST released for the AI RMF in 2025 or 2026?

**Route:** `WEB` — The question pertains to updates or new profiles released by NIST in 2025 or 2026, which are current and time-sensitive topics not covered in the PDF.

### Answer
In 2025 and 2026, NIST has introduced significant updates to the AI Risk Management Framework (AI RMF), focusing on tailored profiles and deeper integration with existing cybersecurity and privacy frameworks. Notably, the introduction of AI RMF Profiles aims to customize the framework for specific sectors, including a concept note for a "Trustworthy AI Profile for Critical Infrastructure." This reflects an evolving understanding of AI systems as critical components that require robust governance and risk management.

Additionally, NIST has aligned the AI RMF with the NIST Cybersecurity Framework (CSF 2.0) and the NIST Privacy Framework, facilitating a more cohesive approach to enterprise risk management. The updates also include operational resources such as playbooks and implementation guides, which provide organizations with practical tools for measuring and managing AI risks effectively. A preliminary draft of the Cyber AI Profile (NIST IR 8596) was released in December 2025, focusing on securing AI systems and addressing AI-enabled cyber threats.

Sources: 
- https://www.ispartnersllc.com/blog/nist-ai-rmf-2025-2026-updates-what-you-need-to-know-about-the-latest-framework-changes
- https://www.ampcuscyber.com/blogs/nist-csf-20-ai-rmf-us-implementation-guide

*Completed in 9.7s*

CrewOutput(raw='In 2025 and 2026, NIST has introduced significant updates to the AI Risk Management Framework (AI RMF), focusing on tailored profiles and deeper integration with existing cybersecurity and privacy frameworks. Notably, the introduction of AI RMF Profiles aims to customize the framework for specific sectors, including a concept note for a "Trustworthy AI Profile for Critical Infrastructure." This reflects an evolving understanding of AI systems as critical components that require robust governance and risk management.\n\nAdditionally, NIST has aligned the AI RMF with the NIST Cybersecurity Framework (CSF 2.0) and the NIST Privacy Framework, facilitating a more cohesive approach to enterprise risk management. The updates also include operational resources such as playbooks and implementation guides, which provide organizations with practical tools for measuring and managing AI risks effectively. A preliminary draft of the Cyber AI Profile (NIST IR 8596) was released in Dec

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# Demo 3: should route to LLM (general knowledge, no retrieval needed)
await ask("In simple terms, what is a large language model?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 610dda65-bec8-4bf9-94fd-7cbb98324780                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: In simple terms, what is a large language model?                                                     │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  ID: 787633f9-f33d-42da-a55f-9e3a7235a3b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: In simple terms, what is a large language model?                                                     │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ROUTE: LLM                                                                                                     │
│  REASON: The question asks for a simple definition, which does not require a specific source document.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze this user question and decide which source should answer it.                                     │
│                                                                                                                 │
│  Question: In simple terms, what is a large language model?                                                     │
│                                                                                                                 │
│  Choose exactly one: PDF, WEB, or LLM.                                                                          │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (5 outputs left out)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 610dda65-bec8-4bf9-94fd-7cbb98324780                                                                       │
│  Final Output: A large language model (LLM) is a type of artificial intelligence that is designed to            │
│  understand and generate human language. These models are trained on vast amounts of text data, allowing them   │
│  to learn patterns, grammar, and context. As a result, LLMs can perform a variety of language-related tasks,    │
│  such as answering questions, summarizing texts, translating languages, and even generating creative content.   │
│                                                                                                                 │
│  LLMs operate using deep learning techniques, particularly neural networks, which enable them to process and    │
│  analyze language in a way that mimics human understanding. They are widely used in applications like           │
│  chatbots, virtual assistants, and content creation tools.                                                      │
│                                                                                                                 │
│  Sources: General knowledge (no retrieval).                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Question
In simple terms, what is a large language model?

**Route:** `LLM` — The question asks for a simple definition, which does not require a specific source document.

### Answer
A large language model (LLM) is a type of artificial intelligence that is designed to understand and generate human language. These models are trained on vast amounts of text data, allowing them to learn patterns, grammar, and context. As a result, LLMs can perform a variety of language-related tasks, such as answering questions, summarizing texts, translating languages, and even generating creative content.

LLMs operate using deep learning techniques, particularly neural networks, which enable them to process and analyze language in a way that mimics human understanding. They are widely used in applications like chatbots, virtual assistants, and content creation tools.

Sources: General knowledge (no retrieval).

*Completed in 4.1s*

CrewOutput(raw='A large language model (LLM) is a type of artificial intelligence that is designed to understand and generate human language. These models are trained on vast amounts of text data, allowing them to learn patterns, grammar, and context. As a result, LLMs can perform a variety of language-related tasks, such as answering questions, summarizing texts, translating languages, and even generating creative content.\n\nLLMs operate using deep learning techniques, particularly neural networks, which enable them to process and analyze language in a way that mimics human understanding. They are widely used in applications like chatbots, virtual assistants, and content creation tools.\n\nSources: General knowledge (no retrieval).', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description='Analyze this user question and decide which source should answer it.\n\nQuestion: In simple terms, what is a large language model?\n\nChoose exactly one: PDF, WEB, or LLM.', name='Analy

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 7. Trace Visualization

The summary table shows how the Router Agent decided for each question. The per-question trace below it shows the path each question took through the agents, and the last cell prints the end of `trace_log.txt` so the raw log is visible in the notebook.

In [11]:
# Summary table: one row per question, showing how the Router decided
rows = "\n".join(
    f"| {i} | {r['question']} | `{r['route']}` | {r['reason']} | {r['seconds']}s |"
    for i, r in enumerate(trace_records, start=1)
)
display(Markdown(
    "## Routing Summary\n\n"
    "| # | Question | Route | Router's reason | Time |\n"
    "|---|---|---|---|---|\n" + rows
))

## Routing Summary

| # | Question | Route | Router's reason | Time |
|---|---|---|---|---|
| 1 | What does the GOVERN function cover in the NIST AI RMF? | `PDF` | The question specifically asks about the content of the NIST AI Risk Management Framework, which is covered in the reference document. | 10.3s |
| 2 | What updates or new profiles has NIST released for the AI RMF in 2025 or 2026? | `WEB` | The question pertains to updates or new profiles released by NIST in 2025 or 2026, which are current and time-sensitive topics not covered in the PDF. | 9.7s |
| 3 | In simple terms, what is a large language model? | `LLM` | The question asks for a simple definition, which does not require a specific source document. | 4.1s |

In [12]:
# Per-question trace: shows the path each question took through the agents
tool_used = {"PDF": "PDFSearchTool (NIST AI RMF vector search)",
             "WEB": "WebSearchTool (Tavily)",
             "LLM": "No tool (general knowledge)"}

for i, r in enumerate(trace_records, start=1):
    display(Markdown(
        f"**Trace {i}:** {r['question']}\n\n"
        f"1. **Router Agent** → chose `{r['route']}` because: *{r['reason']}*\n"
        f"2. **Retriever Agent** → used {tool_used.get(r['route'], 'unknown')}\n"
        f"3. **Answer** returned in {r['seconds']}s\n"
    ))

**Trace 1:** What does the GOVERN function cover in the NIST AI RMF?

1. **Router Agent** → chose `PDF` because: *The question specifically asks about the content of the NIST AI Risk Management Framework, which is covered in the reference document.*
2. **Retriever Agent** → used PDFSearchTool (NIST AI RMF vector search)
3. **Answer** returned in 10.3s


**Trace 2:** What updates or new profiles has NIST released for the AI RMF in 2025 or 2026?

1. **Router Agent** → chose `WEB` because: *The question pertains to updates or new profiles released by NIST in 2025 or 2026, which are current and time-sensitive topics not covered in the PDF.*
2. **Retriever Agent** → used WebSearchTool (Tavily)
3. **Answer** returned in 9.7s


**Trace 3:** In simple terms, what is a large language model?

1. **Router Agent** → chose `LLM` because: *The question asks for a simple definition, which does not require a specific source document.*
2. **Retriever Agent** → used No tool (general knowledge)
3. **Answer** returned in 4.1s


In [13]:
# Display the last part of trace_log.txt so the log is visible in the submitted notebook
with open("trace_log.txt") as f:
    log_text = f.read()
print(log_text[-4000:])   # the last 4,000 characters

cant updates to the AI Risk Management Framework (AI RMF), focusing on tailored profiles and deeper integration with existing cybersecurity and privacy frameworks. Notably, the introduction of AI RMF Profiles aims to customize the framework for specific sectors, including a concept note for a "Trustworthy AI Profile for Critical Infrastructure." This reflects an evolving understanding of AI systems as critical components that require robust governance and risk management.

Additionally, NIST has aligned the AI RMF with the NIST Cybersecurity Framework (CSF 2.0) and the NIST Privacy Framework, facilitating a more cohesive approach to enterprise risk management. The updates also include operational resources such as playbooks and implementation gu
2026-11-11 14:49:34,045 | ROUTE: WEB | REASON: The question pertains to updates or new profiles released by NIST in 2025 or 2026, which are current and time-sensitive topics not covered in the PDF.
2026-11-11 14:49:34,048 | FINAL ANSWER: In 202